In [66]:
# !pip install pytorch_lightning

In [67]:
import random, os, pickle, time
import pandas as pd
import numpy as np
import networkx as nx

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm

from collections import defaultdict


# Set environment variables for reproducibility and safety
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import precision_score, recall_score, accuracy_score

# 1. Configuration & Seeding
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [68]:
NAME = 'movie'

N_CLUSTERS = 15

#### Config

In [69]:
class Config:
    # Đường dẫn dữ liệu
    TCKG_PATH = f"./data/{NAME}/{NAME}_TCKG.csv"  # File CSV chứa đồ thị (như bạn đã gửi)
    # TRAIN_USERS_PATH = "data/train_users.csv" # File chứa danh sách user ID dùng để train
    SAVE_DIR = "./checkpoints"

    # ITEM_START_ID = 2891  # book: 2891
    # ITEM_END_ID = 11584     # book: 11584

    ITEM_START_ID = 1440  # movie
    ITEM_END_ID = 6335

    # Siêu tham số Model
    EMBED_DIM = 64
    HIDDEN_DIM = 256
    HISTORY_LEN = 1   # k' (độ dài lịch sử)
    MAX_PATH_LEN = 3  # K (số bước đi)

    # Siêu tham số Training
    BATCH_SIZE = 32 # Batch lớn giúp RL ổn định hơn
    NUM_EPOCHS = 500
    LEARNING_RATE = 1e-3

    # Thiết bị
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # IDs của các Relation Interaction (Quan trọng cho Reward)
    # Ví dụ: 20=interacted_0, 21=interacted_1, 22=interacted_2
    # INTERACTION_CLUSTER_IDS = [21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35]
    INTERACTION_CLUSTER_IDS = list(range(21, 36))


#### TCKG

In [70]:
class TCKG:
    def __init__(self, tckg_csv_path):
        self.adj_list = defaultdict(list)

        print(f"Loading TCKG from {tckg_csv_path}...")
        df_tckg = pd.read_csv(tckg_csv_path, usecols=['head_id', 'relation_id', 'tail_id'])

        offset = int(df_tckg['relation_id'].max())

        data = df_tckg[['head_id', 'relation_id', 'tail_id']].to_numpy() # Using numpy to speedup
        
        for h, r, t in data:
            h, r, t = int(h), int(r), int(t)

            self.adj_list[h].append((r, t))
            self.adj_list[t].append((r, h))
            # self.adj_list[t].append((r + offset, h)) # Ex: r=5 (watched) -> r_inv=105 (watched_by)

        print(f"TCKG Loaded successfully. Graph construction complete.")

    def get_neighbors(self, node_id):
        return self.adj_list[node_id]

    def get_all_nodes(self):
        return list(self.adj_list.keys())


#### TimeAwareRewardFunction

In [71]:
class TimeAwareRewardFunction(nn.Module):
    def __init__(self, entity_embeddings, relation_embeddings, interaction_cluster_ids, bias_embs=None, temperature= None):
        """
        Args:
            user_embs (nn.Embedding): Embedding của User (e_u)self.max_neighbors = max(len(edges) for edges in self.adj_list.values())
            entity_embs (nn.Embedding): Embedding của Item/Entity (e_v)
            relation_embs (nn.Embedding): Embedding của Relation (dùng để lấy V_U)
            interaction_cluster_ids (list or tensor): Danh sách các Relation ID đại diện cho Time Clusters.
                                                      Ví dụ: [20, 21, 22] ứng với interacted_0, interacted_1...
                                                      Đây chính là tập {V_U^1, ..., V_U^L}
            bias_embs (nn.Embedding, optional): Bias của entity (b_v). Nếu None sẽ tự khởi tạo.
        """
        super(TimeAwareRewardFunction, self).__init__()

        self.entity_embeddings = entity_embeddings
        self.relation_embeddings = relation_embeddings

        # Danh sách ID của các cluster tương tác theo thời gian (V_U)
        # Chuyển thành tensor để tính toán song song
        self.register_buffer('cluster_ids', torch.tensor(interaction_cluster_ids, dtype=torch.long))

        # Entity Bias (b_v) - Eq (11)
        if bias_embs is None:
            num_entities = entity_embeddings.num_embeddings - 1
            self.bias_embeddings = nn.Embedding(num_entities + 1, 1, padding_idx = 0)
            nn.init.zeros_(self.bias_embeddings.weight) # Khởi tạo bias bằng 0
        else:
            self.bias_embeddings = bias_embs

        if temperature is None:
            self.temperature = self.entity_embeddings.embedding_dim ** 0.5
        else:
            self.temperature = temperature

    def calculate_weights(self, history_user_relation_ids):
        """
        Thực hiện Eq (13): Tính trọng số W_hu dựa trên tần suất tương tác chiều Tiến.

        Args:
            history_relation_ids: Tensor (Batch, N) - Các relation ID từ xuat phat tu one user (đã pad 0).

        Returns:
            weights: Tensor (Batch, L) - Vector trọng số thời gian cho L cụm.
        """
        # 1. Chuẩn bị Tensor để Broadcast (So khớp song song trên GPU)
        # hist_expanded: (Batch, History_Len, 1)
        hist_expanded = history_user_relation_ids.unsqueeze(-1)

        # clusters_expanded: (1, 1, L) - Chứa [21, 22, 23, 24]
        clusters_expanded = self.cluster_ids.view(1, 1, -1)

        # 2. So khớp và Đếm (Tử số của Eq 13)
        # matches: (Batch, History_Len, L) -> True/1.0 nếu khớp ID cụm thời gian
        matches = (hist_expanded == clusters_expanded).float()

        # counts: (Batch, L) -> Tổng số lần user tương tác vào mỗi cụm
        counts = matches.sum(dim=1)

        # 3. Tính tổng số tương tác thực tế q (Mẫu số của Eq 13)
        # Bằng cách tính tổng của 'counts', ta tự động lờ đi toàn bộ Padding (ID 0)
        # và các relation ID rác (nếu có), vì chúng không match với forward_clusters.
        # q: (Batch, 1)
        q = counts.sum(dim=1, keepdim=True)

        # 4. Chuẩn hóa để ra trọng số (Thêm 1e-9 để tránh lỗi chia cho 0 nếu user chưa có lịch sử)
        # weights: (Batch, L) -> Đảm bảo tổng các phần tử trên mỗi hàng luôn = 1.0 (hoặc 0)
        weights = counts / (q + 1e-9)

        return weights

    def forward(self, user_ids, item_ids, hist_user_relation_ids):
        """
        Tính Reward Score g_R(v | u)

        Args:
            user_ids: (Batch,)
            item_ids: (Batch,) - Item đích (v_hat) mà Agent dự đoán/dừng lại
            hist_user_relation_ids: (Batch, History_Len) - Lịch sử relation của user

        Returns:
            scores: (Batch,) - Điểm reward
        """
        # --- BƯỚC 1: Lấy Embeddings cơ bản ---
        user_embs = self.entity_embeddings(user_ids)
        item_embs = self.entity_embeddings(item_ids)
        bias_embs = self.bias_embeddings(item_ids).squeeze(-1)

        weights = self.calculate_weights(hist_user_relation_ids)
        weighted_interact_relation_embs = self.relation_embeddings(self.cluster_ids)
        cal_interact_relation_embs = torch.matmul(weights, weighted_interact_relation_embs)

        user_relation_embs = user_embs + cal_interact_relation_embs

        user_relation_embs_norm = F.normalize(user_relation_embs, p=2, dim= 1)
        item_embs_norm = F.normalize(item_embs, p=2, dim= 1) 
        dot_product = torch.sum(user_relation_embs_norm * item_embs_norm, dim=1)
        rewards = dot_product + bias_embs

        return rewards


#### TPRecEnvironment

In [72]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence

class TPRecEnvironment(nn.Module):
    def __init__(self, tckg, entity_embeddings, relation_embeddings, reward_function, max_path_len, history_len, item_start_id, item_end_id):
        """
        Môi trường TPRec được thiết kế chuẩn xác theo kiến trúc Sequence State
        """
        super(TPRecEnvironment, self).__init__()
        self.tckg = tckg
        self.entity_embeddings = entity_embeddings
        self.relation_embeddings = relation_embeddings
        self.max_path_len = max_path_len
        self.history_len = history_len
        self.item_start_id = item_start_id
        self.item_end_id = item_end_id

        # LƯU REWARD FUNCTION ĐƯỢC TRUYỀN VÀO
        self.reward_function = reward_function

        # State tracking
        self.current_entities = None
        self.current_users = None
        self.path_history = None
        self.step_counter = 0

    def reset(self, user_ids):
        """
        Khởi tạo trạng thái s_0 = (u, u, ∅)
        Cập nhật: Nạp thêm target_items để Trọng tài (Môi trường) cầm sẵn đáp án
        """
        batch_size = user_ids.size(0)

        self.current_users = user_ids
        self.current_entities = user_ids

        # History h_k: store (entity, relation) theo chuẩn paper
        self.path_history = torch.zeros((batch_size, self.history_len * 2),     # history_len = k' in paper
                                        dtype=torch.long,
                                        device=user_ids.device)

        self.step_counter = 0
        return self._get_state_embedding()

    def _get_state_embedding(self):
        """
        Kết hợp u, h_k, e_k thành một chuỗi duy nhất đưa vào BLSTM
        """
        # 1. Thêm chiều thời gian (unsqueeze) cho u và e_k
        user_embs = self.entity_embeddings(self.current_users).unsqueeze(1)    # Shape: (B, 1, d)
        entity_embs = self.entity_embeddings(self.current_entities).unsqueeze(1) # Shape: (B, 1, d)

        # 2. Lấy lịch sử e và r
        e_indices = self.path_history[:, 0::2]
        r_indices = self.path_history[:, 1::2]

        e_vecs = self.entity_embeddings(e_indices)   # Shape: (B, L, d)
        r_vecs = self.relation_embeddings(r_indices) # Shape: (B, L, d)

        # 3. Trộn xen kẽ e và r thành chuỗi lịch sử [e1, r2, e2, r3]
        B, L, d = e_vecs.shape
        history_seq = torch.zeros((B, L * 2, d), device=e_vecs.device)
        history_seq[:, 0::2, :] = e_vecs
        history_seq[:, 1::2, :] = r_vecs

        # 4. KẾT NỐI TOÀN BỘ THEO ĐÚNG CÔNG THỨC S_K TRONG PAPER
        full_state_seq = torch.cat([user_embs, history_seq, entity_embs], dim=1) # Shape: (B, Seq_Len, d)

        # Trả về ĐÚNG 1 BIẾN DUY NHẤT đại diện cho S_k
        return full_state_seq

    def get_random_actions(self, epsilon=150):   # used by def get_action_space_batch
        """
        Lấy không gian hành động bằng cách RANDOM SAMPLING các node kề cận.
        """

        raw_actions_list = []

        # 1. Chuyển tensor current_entities hiện tại về Python list để duyệt
        curr_nodes_cpu = self.current_entities.cpu().tolist()
        
        # 2. Vòng lặp lấy ngẫu nhiên láng giềng cho từng user/agent trong batch
        for node in curr_nodes_cpu:
            relation_neighbor_nodes = self.tckg.get_neighbors(node) # relation_neighbor_nodes la cac pari (r1, nb1), (r2, nb2), ...
            
            if len(relation_neighbor_nodes) == 0:
                sampled_actions = [(0, node)] 
            elif len(relation_neighbor_nodes) <= epsilon:
                # Nếu số láng giềng ít hơn sample_size, lấy toàn bộ (không cần random)
                sampled_actions = relation_neighbor_nodes
            else:
                # Nếu quá đông, bốc random đúng `sample_size` láng giềng (không hoàn lại)
                sampled_actions = random.sample(relation_neighbor_nodes, epsilon)
                
            raw_actions_list.append(sampled_actions)

        return raw_actions_list

    def get_action_space_batch(self):
        """
        Lấy không gian hành động cho cả Batch và Padding thành Tensor.
        Tích hợp LUẬT CHỐNG LẶP VÒNG (Cycle Prevention) cực mượt.
        """
        raw_actions_list = self.get_random_actions() # (batch_size, not_fix_size)
        batch_size = len(raw_actions_list)

        lengths = [len(acts) for acts in raw_actions_list] # (batch_size, not_fix_size)
        max_len = max(lengths) if lengths else 0
        if max_len == 0:
            max_len = 1

        device = self.current_entities.device

        r_indices = torch.zeros((batch_size, max_len), dtype=torch.long, device=device) #(batch_size, max_len)
        e_indices = torch.zeros((batch_size, max_len), dtype=torch.long, device=device) #(batch_size, max_len)
        action_mask = torch.zeros((batch_size, max_len), dtype=torch.float, device=device) #(batch_size, max_len)

        # [MỚI] Lấy lịch sử các Entity đã đi qua (Chỉ lấy index chẵn trong path_history)
        if self.path_history is not None:
            visited_entities = self.path_history[:, 0::2].tolist()
        else:
            visited_entities = [[] for _ in range(batch_size)]

        for i, actions in enumerate(raw_actions_list): # i là entity thứ i trong batch
            num_acts = len(actions)
            if num_acts > 0:
                # =====================================================================
                # 1. TẠO DANH SÁCH ĐEN (BLACKLIST) CHO USER i
                # =====================================================================
                blacklist = set(visited_entities[i])            # Các node trong lịch sử
                blacklist.add(self.current_users[i].item())     # Cấm quay về User gốc
                blacklist.add(self.current_entities[i].item())  # Cấm dẫm tại chỗ (Self-loop)

                rs = []
                es = []
                valid_masks = []

                # =====================================================================
                # 2. KIỂM TRA TỪNG HÀNH ĐỘNG VỚI BLACKLIST
                # =====================================================================
                for rel, target_node in actions:
                    rs.append(rel)
                    es.append(target_node)

                    # Nếu target_node nằm trong blacklist -> Mask = 0.0 (Cấm cửa)
                    # Nếu là node mới toanh -> Mask = 1.0 (Cho phép)
                    if target_node in blacklist:
                        valid_masks.append(0.0)
                    else:
                        valid_masks.append(1.0)

                # =====================================================================
                # 3. [QUAN TRỌNG] BẢO HIỂM CHỐNG CRASH (DEAD-END PREVENTION)
                # =====================================================================
                # Xử lý trường hợp Agent đi vào ngõ cụt (Tất cả các ngã rẽ đều đã đi qua)
                # Nếu sum == 0, toàn bộ Mask là 0 -> Điểm -vô cực -> Hàm Softmax sẽ trả về NaN!
                if sum(valid_masks) == 0.0:
                    valid_masks[0] = 1.0 # Bắt buộc mở khóa 1 đường bất kỳ để mạng nơ-ron không bị nổ (Crash)

                # 4. Gán dữ liệu vào Tensor
                r_indices[i, :num_acts] = torch.tensor(rs, device=device) # (batch_size, num_acts)
                e_indices[i, :num_acts] = torch.tensor(es, device=device) # (batch_size, num_acts)
                action_mask[i, :num_acts] = torch.tensor(valid_masks, dtype=torch.float, device=device) # (batch_size, num_acts)

        # 5. Lấy vector nhúng từ ID
        r_emb = self.relation_embeddings(r_indices) # (batch_size, num_acts, embed_dim)
        e_emb = self.entity_embeddings(e_indices) # (batch_size, num_acts, embed_dim)

        # CHỐT: Ghép theo đúng định nghĩa Toán học Action = (Relation, Entity)
        action_embs = torch.cat([r_emb, e_emb], dim=-1) # (batch_size, num_acts, embed_dim * 2)

        return action_embs, action_mask, raw_actions_list

    def step(self, actions):    # used  by def step_with_indices
        """
        Transition function (Eq 9): Chuyển trạng thái sang bước k+1
        """
        device = self.current_entities.device
        batch_size = len(actions)

        rels_list = [a[0] for a in actions]
        ents_list = [a[1] for a in actions]

        next_relations = torch.tensor(rels_list, dtype=torch.long, device=device)
        next_entities = torch.tensor(ents_list, dtype=torch.long, device=device)

        # Lấy Node hiện tại làm điểm xuất phát (e_{k-1})
        curr_e = self.current_entities.unsqueeze(1)
        new_r = next_relations.unsqueeze(1)

        # Nối lại theo chuẩn Paper: [e_{k-1}, r_k]
        new_entry = torch.cat([curr_e, new_r], dim=1)

        history_shifted = self.path_history[:, 2:]
        self.path_history = torch.cat([history_shifted, new_entry], dim=1)

        self.current_entities = next_entities #tensor([8673, 8614])
        self.step_counter += 1
        done = (self.step_counter >= self.max_path_len)

        return self._get_state_embedding(), done

    def step_with_indices(self, action_indices, raw_actions_list):
        selected_real_actions = []
        batch_size = len(action_indices)

        for i in range(batch_size):
            idx = action_indices[i].item()

            # [ĐÃ SỬA TÊN] Dùng 'node_acts' thay vì 'user_acts'
            node_acts = raw_actions_list[i]

            if len(node_acts) > 0:
                idx = min(idx, len(node_acts) - 1)
                real_action = node_acts[idx]
            else:
                curr_node = self.current_entities[i].item()
                real_action = (0, curr_node) # Dead-end: Tự trỏ về chính nó

            selected_real_actions.append(real_action)

        return self.step(selected_real_actions)

    def get_reward(self):
        """
        Gọi khi done=True (Hoặc ở các bước lẻ).
        Tính Time-aware Reward g_R(v|u). Nếu không phải Item -> Reward = 0.0

        Lưu ý: tham số truyền vào nên là 1 Tensor chứa TOÀN BỘ các ID của Item có trong Đồ thị.
        """
        user_ids = self.current_users
        item_ids = self.current_entities

        batch_history_relations = []

        for user_id in user_ids.tolist():
            batch_neighbors = self.tckg.get_neighbors(user_id)
            user_rels = [rel for (rel, tail) in batch_neighbors]
            user_rels_tensor = torch.tensor(user_rels, dtype=torch.long)
            batch_history_relations.append(user_rels_tensor)

        history_relation_ids = pad_sequence(
            batch_history_relations,
            batch_first=True,
            padding_value=0
        ).to(self.current_users.device)

        # 1. SOFT REWARD: Tính điểm dẫn đường cho TOÀN BỘ node hiện tại (bất chấp là Item hay Entity)
        # rewards có shape: (Batch_size,)
        rewards = self.reward_function(user_ids, item_ids, history_relation_ids)

        # =====================================================================
        # 2. [MỚI] MASKING: TẮT REWARD VỀ 0.0 NẾU KHÔNG PHẢI LÀ ITEM
        # =====================================================================
        is_item_mask = (item_ids >= self.item_start_id) & (item_ids <= self.item_end_id)

        rewards = torch.where(is_item_mask, rewards, torch.zeros_like(rewards))

        return rewards

#### TPRecPolicy

In [73]:
import torch
import torch.nn as nn

class TPRecPolicy(nn.Module):
    def __init__(self, embed_dim, hidden_dim, dropout):
        super(TPRecPolicy, self).__init__()

        # BLSTM giờ đây chỉ cần nhận input_size = embed_dim (không cần nhân 2 nữa)
        self.blstm = nn.LSTM(input_size=embed_dim,
                             hidden_size=hidden_dim,
                             num_layers=1,
                             batch_first=True,
                             bidirectional=True)

        # W1 giờ đây chỉ cần nhận đầu ra của BLSTM
        self.W1 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.dropout = nn.Dropout(dropout)

        self.Wa = nn.Linear(hidden_dim, embed_dim * 2)
        self.Wc = nn.Linear(hidden_dim, 1)

    def forward(self, full_state_seq, action_embs, action_mask):
        # 1. Đưa toàn bộ s_k vào BLSTM như công thức (15)
        lstm_out, _ = self.blstm(full_state_seq)

        # # Lấy trạng thái ở bước thời gian cuối cùng đại diện cho toàn bộ chuỗirelation_idrelation_id
        # lstm_last = lstm_out[:, -1, :]

        # Vì bidirectional=True, hidden_dim này thực chất là 2 phần ghép lại
        half_dim = self.blstm.hidden_size

        # Lấy trạng thái Tóm tắt Chiều đi tới (Ở index cuối cùng: -1)
        forward_last = lstm_out[:, -1, :half_dim]

        # Lấy trạng thái Tóm tắt Chiều đi lùi (Ở index đầu tiên: 0)
        backward_last = lstm_out[:, 0, half_dim:]

        # Ghép 2 tóm tắt này lại thành một Context Vector hoàn hảo
        lstm_last = torch.cat([forward_last, backward_last], dim=-1)

        # 2. Tính x_k theo đúng phương trình
        x_k = self.dropout(torch.sigmoid(self.W1(lstm_last)))

        # --- (Phần tính Actor và Critic giữ nguyên) ---
        query = self.Wa(x_k).unsqueeze(1)
        scores = torch.sum(query * action_embs, dim=-1)
        scores = scores.masked_fill(action_mask.bool() == False, float('-inf'))
        probs = torch.softmax(scores, dim=-1)

        value_baseline = self.Wc(x_k).squeeze(-1)

        return probs, value_baseline

#### TPRecLightningModel

In [ ]:
import pytorch_lightning as pl
import torch, math
import torch.optim as optim

class TPRecLightningModel(pl.LightningModule):
    def __init__(self, interaction_cluster_ids, env, policy_net, learning_rate):
        super().__init__()

        # Lưu lại hyperparameter (tự động log vào tensorboard nếu dùng)
        self.save_hyperparameters(ignore=['env', 'policy_net'])

        self.interaction_cluster_ids = interaction_cluster_ids
        self.env = env
        self.policy_net = policy_net
        self.learning_rate = learning_rate

    def setup(self, stage=None):
        self.val_true_items = self.trainer.datamodule.val_true_items

    @staticmethod
    def hit_at_k(pred_items, true_items):
        hits = 0
        for pred, true in zip(pred_items, true_items):
            # Count if any true item is in top-k predictions
            if len(set(pred) & set(true)) > 0:
                hits += 1
        return hits / len(true_items)

    @staticmethod
    def ndcg_at_k(pred_items, true_items):
        ndcg = 0.0
        for pred, true in zip(pred_items, true_items):
            gains = []
            for idx, item in enumerate(pred):
                gains.append(1 if item in true else 0)
            ideal_gains = [1] * len(true)
            dcg = sum(g / math.log2(i+2) for i, g in enumerate(gains))
            idcg = sum(g / math.log2(i+2) for i, g in enumerate(ideal_gains))
            ndcg += dcg / idcg if idcg > 0 else 0
        return ndcg / len(true_items)

    @staticmethod
    def recall_at_k(pred_items, true_items):
        recall = 0.0
        for pred, true in zip(pred_items, true_items):
            recall += len(set(pred) & set(true)) / len(true)
        return recall / len(true_items)

    @staticmethod
    def precision_at_k(pred_items, true_items):
        precision = 0.0
        for pred, true in zip(pred_items, true_items):
            k = len(pred)
            precision += len(set(pred) & set(true)) / k
        return precision / len(true_items)
    
    def forward(self, full_state_seq, action_embs, action_mask):
        """
        Đạo diễn Lightning chuyển tiếp ĐÚNG 3 tham số cho Diễn viên Policy Network
        """
        probs, value_baseline =  self.policy_net(full_state_seq, action_embs, action_mask)
        return probs, value_baseline

    def _compute_loss(self, batch_users):
        saved_log_probs = []
        saved_values = []

        full_state_seq = self.env.reset(batch_users)

        for t in range(self.env.max_path_len):
            action_embs, action_mask, raw_actions = self.env.get_action_space_batch()
            probs, value_baseline = self(full_state_seq, action_embs, action_mask) # prob (batch_size, num_act), xac suat de chon cac action. value_baseline (batch_size): ky vong se nhan duoc bao nhieu reward

            m = torch.distributions.Categorical(probs)
            action_indices = m.sample()

            # Thực hiện bước đi
            full_state_seq, done = self.env.step_with_indices(action_indices, raw_actions)

            saved_log_probs.append(m.log_prob(action_indices))
            saved_values.append(value_baseline)

        # =====================================================================
        # TÍNH TOÁN LỢI TỨC G THEO ĐỊNH LÝ BELLMAN
        # =====================================================================
        final_reward = self.env.get_reward().detach()

        # Tự động tạo mảng saved_rewards: [0, 0, ..., final_reward]
        # Dùng List Comprehension để tạo các tensor 0 độc lập, tránh tham chiếu bộ nhớ
        saved_rewards = [torch.zeros_like(final_reward) for _ in range(self.env.max_path_len - 1)]
        saved_rewards.append(final_reward)

        gamma = 0.99
        returns = []

        G = torch.zeros_like(saved_rewards[0])

        for R_step in reversed(saved_rewards):
            G = R_step + gamma * G
            returns.insert(0, G)

        policy_loss = 0
        value_loss = 0

        # CÔNG THỨC 18: Tính Loss REINFORCE
        for G_val, log_prob, value_baseline in zip(returns, saved_log_probs, saved_values):
            advantage = G_val - value_baseline.detach()

            step_policy_loss = -log_prob * advantage
            policy_loss += step_policy_loss.mean()

            step_value_loss = torch.nn.functional.mse_loss(value_baseline, G_val)
            value_loss += step_value_loss

        total_loss = policy_loss + value_loss
        total_episode_reward = sum(saved_rewards)

        return total_loss, total_episode_reward

    def training_step(self, batch, batch_idx):
        """
        Phiên bản Thử nghiệm: Thưởng dọc đường (Multi-hop) nhưng BỎ QUA Bước 1.
        Agent chỉ được nhận Reward khi đến các trạm Item ở t=2, t=4...
        """
        batch_users, _ = batch[0], batch[1]

        train_loss, train_reward = self._compute_loss(batch_users)
        self.log('train_reward', train_reward.mean(), prog_bar=True, on_step=False, on_epoch=True)
        self.log('train_loss', train_loss, prog_bar=False, on_step=False, on_epoch=True)

        return train_loss
    
    def validation_step(self, batch, batch_idx):            ### Tinh val_loss, val_reward
        """
        Phiên bản Thử nghiệm: Thưởng dọc đường (Multi-hop) nhưng BỎ QUA Bước 1.
        Agent chỉ được nhận Reward khi đến các trạm Item ở t=2, t=4...
        """
        batch_users, _ = batch[0], batch[1]

        val_loss, val_reward = self._compute_loss(batch_users)
        self.log('val_reward', val_reward.mean(), prog_bar=True, on_step=False, on_epoch=True)
        self.log('val_loss', val_loss, prog_bar=False, on_step=False, on_epoch=True)


    def training_step(self, batch, batch_idx):        ### Tinh val_hr, ....
        """
        Vòng lặp Đánh giá (Validation) mô phỏng chính xác Section 3.4 của TPRec.
        Bao gồm: Val Loss, Val Reward, và Re-ranking bằng Công thức (20) để tính HR/NDCG.
        """
        batch_users = batch[0]
        target_items = batch[1]
        W_T_hat = batch[2] # Shape: (Batch_size, N_Clusters)
        batch_relations = batch[3]

        batch_size = batch_users.size(0)

        num_paths = 20 #K = 20 nghĩa là ta muốn agent đi 20 đường khác nhau để tìm ra tối đa 20 items ứng viên.

        repeated_users = batch_users.repeat_interleave(num_paths) # nhân bản thanh (batch_size * num_paths)

        # Reset môi trường với batch đã được nhân bản
        full_state_seq = self.env.reset(repeated_users) # (batch_size * num_paths, 4, embed_dim). 4 = 1 (user_emb) + 2 (history_rel & history neighbor), 1 entity

        for t in range(self.env.max_path_len):
            action_embs, action_mask, raw_actions = self.env.get_action_space_batch() #(batch_size * num_paths, , )
            probs, value_baseline = self(full_state_seq, action_embs, action_mask)

            action_indices = torch.multinomial(probs, num_samples=1).squeeze(-1) #nhân bản thành 20 bản sao, hàm multinomial sẽ cho phép các bản sao này rẽ sang các hướng khác nhau tại mỗi bước nhảy

            full_state_seq, done = self.env.step_with_indices(action_indices, raw_actions)

        final_nodes = self.env.current_entities.view(-1) # Shape: (batch_size * num_paths)
        

        candidate_items = final_nodes.view(batch_size, num_paths) # (batch_size, num_paths). Bây giờ candidate_items chứa 20 đường dẫn đến 20 item (có thể trùng lặp) cho mỗi user


        e_u = self.env.entity_embeddings(batch_users) # Shape: (Batch, Embed_Dim)
        e_V_hat = self.env.entity_embeddings(candidate_items) # (batch_size, num_paths, embed_dim)

        cluster_ids_tensor = torch.tensor(
            self.interaction_cluster_ids,
            dtype=torch.long,
            device=self.device # Đảm bảo Tensor nằm cùng chỗ với Model
        )

        # 2. Truyền Tensor vào lớp Embedding
        V_U_T = self.env.relation_embeddings(cluster_ids_tensor) # (num_cluster, embed_dim)
        r_W_T = torch.matmul(W_T_hat.float(), V_U_T) # (Batch_size, Embed_Dim)

        # Tạo Query Vector theo Công thức 20: (e_u + r_W_T)
        query_vector = e_u + r_W_T # Shape: (Batch, Embed_Dim)

        query_vector_norm = F.normalize(query_vector, p=2, dim=1) # (Batch, Embed_Dim)
        e_V_hat_norm = F.normalize(e_V_hat, p=2, dim=1) # (Batch, num_paths, Embed_Dim)

        # Áp dụng Công thức (20): Phép nhân vô hướng (Dot Product) với tập ứng viên e_V_hat
        scores = torch.sum(query_vector_norm.unsqueeze(1) * e_V_hat_norm, dim=-1) # Shape: (Batch, num_paths)


        topk_relative_indices = torch.topk(scores, k=10, dim=1).indices # (batch, k_topk)

        # 4. Map ngược ra ID thực tế bằng cách truyền thẳng vào pool_ids_tensor
        # Lưu ý: pool_ids_tensor chính là torch.tensor(self.item_pool) mà bạn đã tạo ở trên
        # Shape: [batch_size, k]
        topk_actual_ids = torch.gather(candidate_items, dim=1, index=topk_relative_indices) #(batch, k_topk)

        # 5. (Tùy chọn) Ép về Python list dạng [ [id1, id2...], [id3, id4...] ]
        # Để đưa vào các hàm tính Hit@10 hoặc NDCG@10relation_embs
        topk_items = topk_actual_ids.tolist()

            # 2. Ép kiểu set thành list trực tiếp, gom gọn trong 1 dòng
        # true_items = [list(self.val_true_items[(u, r)]) for u, r in zip(batch_users.tolist(), batch_relations.tolist())]
        true_items = []

        for u, r in zip(batch_users.tolist(), batch_relations.tolist()):
            # Dùng list() ép kiểu trực tiếp thay vì [item for item in ...]
            items_list = list(self.val_true_items[(u, r)])
            true_items.append(items_list)

        # Compute metrics for this k
        hit = self.hit_at_k(topk_items, true_items)
        ndcg = self.ndcg_at_k(topk_items, true_items)
        recall = self.recall_at_k(topk_items, true_items)
        precision = self.precision_at_k(topk_items, true_items)\

        self.log(f"val_hit@10", hit, prog_bar=True)
        self.log(f"val_recall@10", recall, prog_bar=True)
        self.log(f"val_precision@10", precision, prog_bar=True)
        self.log(f"val_ndcg@10", ndcg, prog_bar=True)

    def configure_optimizers(self):
        # Lọc ra CHỈ những tham số đang requires_grad=True (Mạng Policy)
        # Giúp tránh lỗi tối ưu hóa các tham số đã bị freeze (như Embeddings)
        trainable_params = filter(lambda p: p.requires_grad, self.parameters())

        optimizer = optim.Adam(trainable_params, lr=self.learning_rate)
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.9)

        return [optimizer], [scheduler]

### Load Dataset

In [75]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from collections import defaultdict
import os

class InteractionDataset(Dataset):
    def __init__(self, df, n_clusters):
        """Nhận vào một DataFrame và chuyển đổi các cột cần thiết thành mảng NumPy"""
        self.user_ids = df['user_id'].values
        self.item_ids = df['entity_id'].values
        self.relation_ids = 21 + df['cluster_label'].values # 21 là ID bắt đầu
        
        prob_cols = [f'prob_cluster_{i}' for i in range(n_clusters)]
        self.multi_cluster_weights = df[prob_cols].values

    def __len__(self):
        return len(self.user_ids)

    def __getitem__(self, idx):
        user_id = torch.tensor(self.user_ids[idx], dtype=torch.long)
        item_id = torch.tensor(self.item_ids[idx], dtype=torch.long)
        multi_cluster_weight = torch.tensor(self.multi_cluster_weights[idx], dtype=torch.float32)
        relation_id = torch.tensor(self.relation_ids[idx], dtype=torch.long) 

        return user_id, item_id, multi_cluster_weight, relation_id

class InteractionDataModule(pl.LightningDataModule):
    def __init__(self, data_dir: str, dataset_name: str, n_clusters: int, batch_size: int, num_workers: int):
        """
        Khai báo các siêu tham số (Hyperparameters) cho DataModule
        """
        super().__init__()
        self.data_dir = data_dir
        self.dataset_name = dataset_name
        self.n_clusters = n_clusters
        self.batch_size = batch_size
        self.num_workers = num_workers
        
    def _create_ground_truth(self, df):
        """Hàm helper để gom nhóm Ground Truth từ bất kỳ DataFrame nào"""
        grouped_dict = df.groupby([df['user_id'], df['cluster_label'] + 21])['entity_id'].apply(set).to_dict()
        return defaultdict(set, grouped_dict)
    
    def setup(self, stage=None):
        """
        Hàm này tự động được gọi trên mỗi GPU (nếu dùng Multi-GPU).
        Thực hiện các tác vụ: Đọc file, chia split, khởi tạo Dataset.
        """
        # Đường dẫn cơ sở
        base_path = os.path.join(self.data_dir, self.dataset_name)
        
        # Nhánh 1: Kích hoạt khi gọi trainer.fit()
        if stage == 'fit':
            train_df = pd.read_csv(f'{base_path}/{self.dataset_name}_gmm_train_interactions.csv')
            val_df = pd.read_csv(f'{base_path}/{self.dataset_name}_gmm_val_interactions.csv')

            # TẠO GROUND TRUTH Ở ĐÂY TRƯỚC KHI ĐƯA VÀO DATASET
            self.train_true_items = self._create_ground_truth(train_df)
            self.val_true_items = self._create_ground_truth(val_df)
            
            self.train_dataset = InteractionDataset(train_df, self.n_clusters)
            self.val_dataset = InteractionDataset(val_df, self.n_clusters)
            
        # Nhánh 2: Kích hoạt khi CHỈ gọi trainer.validate() độc lập
        if stage == 'validate':
            val_df = pd.read_csv(f'{base_path}/{self.dataset_name}_gmm_val_interactions.csv')
            self.val_true_items = self._create_ground_truth(val_df)
            
            self.val_dataset = InteractionDataset(val_df, self.n_clusters)

        # Nhánh 3: Kích hoạt khi gọi trainer.test()
        if stage == 'test' or stage is None:
            # test_df = pd.read_csv(...)
            # self.test_dataset = InteractionDataset(...)
            pass

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset, 
            batch_size=self.batch_size, 
            shuffle=True, 
            num_workers=self.num_workers,
            pin_memory=True # Giúp chuyển data từ RAM lên GPU nhanh hơn
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset, 
            batch_size=self.batch_size, 
            shuffle=False, 
            num_workers=self.num_workers,
            pin_memory=True
        )

    def test_dataloader(self):
        # return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)
        pass

datamodule = InteractionDataModule(
    data_dir='./data', 
    dataset_name='movie', # Tương ứng với biến NAME của bạn
    n_clusters=N_CLUSTERS,                       # Tương ứng với N_CLUSTERS
    batch_size=256,
    num_workers = 0
)

### Main function

In [80]:
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

cfg = Config()

tckg = TCKG(cfg.TCKG_PATH)


pickle_file_path = f'./pickle/movie_KGCN_embeddings_2026-03-07_11-06-40.pkl'

with open(pickle_file_path, 'rb') as f:
    saved_data = pickle.load(f)

pretrained_ent = saved_data['entity_embeddings']
pretrained_rel = saved_data['relation_embeddings']

if isinstance(pretrained_ent, np.ndarray):
    ent_tensor = torch.tensor(pretrained_ent, dtype=torch.float32)
    rel_tensor = torch.tensor(pretrained_rel, dtype=torch.float32)
else:
    # Nếu data đã là Tensor sẵn:
    ent_tensor = pretrained_ent.clone().detach().float()
    rel_tensor = pretrained_rel.clone().detach().float()

# 3. Nạp vào nn.Embedding
# freeze=False: Cho phép RL Agent tiếp tục cập nhật (fine-tune) vector trong lúc tìm đường
# freeze=True: Khóa cứng vector, RL Agent chỉ học Policy Network (Học nhanh hơn, chống overfit)
entity_embeddings = nn.Embedding.from_pretrained(ent_tensor, freeze=False, padding_idx=0)
relation_embeddings = nn.Embedding.from_pretrained(rel_tensor, freeze=False, padding_idx=0)


reward_func = TimeAwareRewardFunction(
    entity_embeddings=entity_embeddings,    # Chia sẻ trọng số với Env
    relation_embeddings=relation_embeddings,
    interaction_cluster_ids=cfg.INTERACTION_CLUSTER_IDS,
    bias_embs=None, # Tự tạo bias mới
    temperature= 1
)


env = TPRecEnvironment(
    tckg=tckg,
    entity_embeddings=entity_embeddings,
    relation_embeddings=relation_embeddings,
    reward_function=reward_func, # Inject reward vào env
    max_path_len=cfg.MAX_PATH_LEN,
    history_len=cfg.HISTORY_LEN,
    item_start_id=cfg.ITEM_START_ID,
    item_end_id=cfg.ITEM_END_ID
)

policy_net = TPRecPolicy(
        embed_dim=cfg.EMBED_DIM,      # Ví dụ: 64
        hidden_dim=cfg.HIDDEN_DIM,    # Ve([256, 4, 512])í dụ: 128
        dropout=0.3                   # Tỉ lệ dropout giúp chống Overfit (theo paper)
    )


lightning_model = TPRecLightningModel(
    interaction_cluster_ids=cfg.INTERACTION_CLUSTER_IDS,
    env=env,
    policy_net=policy_net,
    learning_rate=cfg.LEARNING_RATE
)

checkpoint_callback = ModelCheckpoint(
    dirpath=cfg.SAVE_DIR,
    filename='tprec-best-{epoch:02d}-{train_reward:.4f}',
    monitor='train_reward',
    mode='max', # Lưu model có reward lớn nhất
    save_top_k=1,
    save_last=True # Lưu thêm model ở epoch cuối cùng để phòng hờ
)


early_stop_callback = EarlyStopping(
      monitor='train_reward',   # Phải cùng tên với biến monitor ở Checkpoint
      min_delta=0.001,       # Sự thay đổi tối thiểu để được tính là "có cải thiện"
      patience=20,           # Sức chịu đựng: Cho phép mô hình dậm chân tại chỗ tối đa 10 Epoch
      verbose=True,          # Bật in thông báo ra màn hình khi Early Stop kích hoạt
      mode='max'             # 'max' vì ta muốn chỉ số HR@10/Reward càng lớn càng tốt
  )


trainer = pl.Trainer(
    max_epochs=cfg.NUM_EPOCHS,
    accelerator="auto", # Tự động tìm và dùng GPU nếu có
    devices=1,
    gradient_clip_val=1.0, # Tự động áp dụng Gradient Clipping
    callbacks=[checkpoint_callback, early_stop_callback],
    enable_progress_bar=True,
    num_sanity_val_steps=0,
    # limit_val_batches=0,       # 1. Bỏ qua vòng lặp validation ở cuối mỗi epoch
)


# trainer.fit(
#         model=lightning_model,
#         datamodule= datamodule
#     )


Loading TCKG from ./data/movie/movie_TCKG.csv...


GPU available: False, used: False
TPU available: False, using: 0 TPU cores


TCKG Loaded successfully. Graph construction complete.


In [91]:
path = "/home/hp/Study/07. Luan Van/03. TPRec/04. RL/checkpoints/tprec-best-epoch=22-train_reward=0.7455.ckpt"
model = TPRecLightningModel.load_from_checkpoint(path, env=env, policy_net=policy_net, weights_only= False)
trainer.validate(model= model, datamodule= datamodule)


Validation DataLoader 0: 100%|██████████| 6/6 [00:01<00:00,  5.40it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        val_loss           0.006716858595609665
       val_reward           0.7343387603759766
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'val_reward': 0.7343387603759766, 'val_loss': 0.006716858595609665}]